# Build and inspect one network

Choose one source in the settings cell, then load or rebuild that named result. The notebook validates the canonical PyPSA NetCDF against its GeoParquet bundle and writes one static map and one interactive map for the selected network.

Demand is attached later. Inferred results are connectivity proxies, not confirmed electrical networks. `inferred-data` includes only the reviewed generator records whose modelling fields are complete; `inferred-osm` keeps OSM generator sites as topology terminals because OSM does not provide reviewed operating capacity.

## Settings

In [ ]:
import json
import math
import os
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import pandas as pd
import plotly.graph_objects as go
import pypsa
from IPython.display import FileLink, Image, display

from mu_star_energy.network_source import build_network
from mu_star_energy.osm import OSMDownloadRequired
from mu_star_energy.paths import network_output_dir, processed_energy_dir

# Select exactly one input source: "base", "inferred-osm", or "inferred-data".
NETWORK_SOURCE = "inferred-data"
INFERRED_REGION = "mauritius-rodrigues"
INFERRED_NETWORK_TYPE = "all"

OUTPUT_NAME = (
    "base-mauritius"
    if NETWORK_SOURCE == "base"
    else f"{NETWORK_SOURCE}-{INFERRED_REGION}"
)

# Safe defaults load cached/generated data. Enable these deliberately when needed.
REBUILD_NETWORK = False
ALLOW_DOWNLOAD = False

# High-pass threshold used to identify VIIRS nightlight targets.
NIGHTLIGHT_THRESHOLD = 0.1
NIGHTLIGHT_SUPPORT_DISTANCE_M = 1000.0
MAX_ANCHOR_DISTANCE_M = 1000.0

# Data-stage convention:
#   1-processed/energy/provided/  -> standardised inputs only.
#   2-out/energy/networks/<name>/ -> the whole build (a model output): the
#       PyPSA .nc, metadata, GeoParquet bundle, review CSV tables and the maps.
PROVIDED_DIR = processed_energy_dir() / "provided"
NETWORK_RESULTS_ROOT = network_output_dir()
HUMAN_TABLE_OUTPUT_ROOT = network_output_dir()
RESULT_DIR = NETWORK_RESULTS_ROOT / OUTPUT_NAME
GEOPARQUET_DIR = RESULT_DIR / "geoparquet"
VISUALISATION_DIR = RESULT_DIR / "visualisations"

NETWORK_PATH = RESULT_DIR / f"{OUTPUT_NAME}.nc"
METADATA_PATH = RESULT_DIR / f"{OUTPUT_NAME}_metadata.json"
NODES_PATH = GEOPARQUET_DIR / f"{OUTPUT_NAME}-nodes.geoparquet"
EDGES_PATH = GEOPARQUET_DIR / f"{OUTPUT_NAME}-edges.geoparquet"
MANIFEST_PATH = GEOPARQUET_DIR / f"{OUTPUT_NAME}-spatial-manifest.json"

print("Selected source:", NETWORK_SOURCE)
print("Build directory (2-out):", RESULT_DIR)


## Load or rebuild the selected result

For `base`, the build reads the reviewed substations, routed CEB line geometry and generator table under `data/1-processed/energy/provided`.

For `inferred-osm`, cached OSM substations, plants and generators are known power terminals. For `inferred-data`, only the reviewed input substations and generator sites are used. Both methods use VIIRS nightlight targets to retain a dense, cyclic OSM road subnetwork within the configured support distance. The reviewed-data result also preserves the CEB backbone. A member island without a known power asset receives a labelled provisional root.


In [ ]:
if NETWORK_SOURCE not in {"base", "inferred-osm", "inferred-data"}:
    raise ValueError(
        "NETWORK_SOURCE must be 'base', 'inferred-osm', or 'inferred-data'"
    )

if REBUILD_NETWORK:
    try:
        outputs = build_network(
            NETWORK_SOURCE,
            input_dir=PROVIDED_DIR,
            output_dir=NETWORK_RESULTS_ROOT,
            region=INFERRED_REGION if NETWORK_SOURCE != "base" else None,
            output_name=OUTPUT_NAME,
            overwrite=True,
            allow_download=ALLOW_DOWNLOAD,
            network_type=INFERRED_NETWORK_TYPE,
            nightlight_threshold=NIGHTLIGHT_THRESHOLD,
            nightlight_support_distance_m=NIGHTLIGHT_SUPPORT_DISTANCE_M,
            max_anchor_distance_m=MAX_ANCHOR_DISTANCE_M,
            export_root=HUMAN_TABLE_OUTPUT_ROOT,
        )
    except OSMDownloadRequired as err:
        raise RuntimeError(
            "A required OSM cache is missing. Set ALLOW_DOWNLOAD=True only "
            "when you intend to fetch and cache it, then rerun this cell."
        ) from err

    NETWORK_PATH = outputs.network
    METADATA_PATH = outputs.metadata
    NODES_PATH = outputs.spatial_nodes
    EDGES_PATH = outputs.spatial_edges
    MANIFEST_PATH = outputs.spatial_manifest
    action = "rebuilt"
else:
    expected = (
        NETWORK_PATH,
        METADATA_PATH,
        NODES_PATH,
        EDGES_PATH,
        MANIFEST_PATH,
    )
    missing = [path for path in expected if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "Missing selected network artifacts:\n- "
            + "\n- ".join(str(path) for path in missing)
            + "\nSet REBUILD_NETWORK=True to reproduce them."
        )
    action = "loaded"

network = pypsa.Network(NETWORK_PATH)
metadata = json.loads(METADATA_PATH.read_text())
manifest = json.loads(MANIFEST_PATH.read_text())
nodes = gpd.read_parquet(NODES_PATH).set_index("bus_id", drop=False)
edges = gpd.read_parquet(EDGES_PATH).set_index("line_id", drop=False)

if nodes.crs is None or nodes.crs.to_epsg() != 4326:
    raise ValueError("Node GeoParquet must use EPSG:4326")
if edges.crs is None or edges.crs.to_epsg() != 4326:
    raise ValueError("Edge GeoParquet must use EPSG:4326")
if set(nodes.index.astype(str)) != set(network.buses.index.astype(str)):
    raise ValueError("GeoParquet node IDs do not match NetCDF buses")
if set(edges.index.astype(str)) != set(network.lines.index.astype(str)):
    raise ValueError("GeoParquet edge IDs do not match NetCDF lines")
node_ids = set(nodes.index.astype(str))
if not set(edges["bus0"].astype(str)).issubset(node_ids):
    raise ValueError("GeoParquet contains an unknown bus0 endpoint")
if not set(edges["bus1"].astype(str)).issubset(node_ids):
    raise ValueError("GeoParquet contains an unknown bus1 endpoint")

length_check = metadata.get(
    "line_length_validation",
    metadata.get("road_envelope_line_length_validation", {}),
)
summary = pd.Series(
    {
        "action": action,
        "source": NETWORK_SOURCE,
        "methodology": metadata.get("methodology"),
        "buses": len(network.buses),
        "lines": len(network.lines),
        "generators": len(network.generators),
        "line length (km)": float(edges["length_km"].sum()),
        "CEB length-check status": length_check.get("status"),
        "CEB relative difference": length_check.get("relative_difference"),
        "OSM road-envelope edges": metadata.get("road_envelope_edges"),
        "power-asset source": metadata.get("power_asset_source"),
        "substation terminals": metadata.get("substation_roots"),
        "generator terminals": metadata.get("generator_roots"),
    },
    name=OUTPUT_NAME,
)
display(summary.to_frame("value"))
print(f"{action.capitalize()} and validated: {NETWORK_PATH}")


## Static map

The map reads the routed GeoParquet edge geometry, not straight bus-to-bus chords. Lines are coloured by a network attribute (`source` by default; set `LINE_COLOUR_BY = "v_nom_kv"` for voltage), not by region. Every node is a bus distinguished by kind (substation, generator, junction). When a result spans several regions, one panel is drawn per region using the same colours throughout.


In [ ]:
# Lines are coloured by a network attribute, not by region. Set LINE_COLOUR_BY
# to "source" or "v_nom_kv". Every node is a bus, distinguished by its kind.
LINE_COLOUR_BY = "source"


def line_segments(frame):
    segments = []
    for geometry in frame.geometry:
        parts = [geometry] if geometry.geom_type == "LineString" else geometry.geoms
        for part in parts:
            if part.geom_type == "LineString":
                segments.append(
                    [(float(point[0]), float(point[1])) for point in part.coords]
                )
    return segments


def set_map_extent(axis, frame):
    min_x, min_y, max_x, max_y = frame.total_bounds
    x_pad = max((max_x - min_x) * 0.04, 0.005)
    y_pad = max((max_y - min_y) * 0.04, 0.005)
    axis.set_xlim(min_x - x_pad, max_x + x_pad)
    axis.set_ylim(min_y - y_pad, max_y + y_pad)
    axis.set_aspect("equal", adjustable="box")
    axis.set_xlabel("Longitude")
    axis.set_ylabel("Latitude")
    axis.grid(alpha=0.15)


def line_category(frame, column=LINE_COLOUR_BY):
    """Return a readable category label per line for the chosen attribute."""
    if column not in frame or frame.empty:
        return pd.Series(["unspecified"] * len(frame), index=frame.index)
    values = frame[column]
    if column.endswith("_kv") or column.startswith("v_nom"):
        return values.map(lambda v: "unspecified" if pd.isna(v) else f"{float(v):g} kV")
    return values.fillna("unspecified").astype(str)


def node_kind(value):
    """Collapse node kinds to one shared vocabulary of bus kinds."""
    kind = "junction" if pd.isna(value) else str(value)
    if kind == "distribution_node":
        kind = "junction"
    return kind if kind in NODE_STYLE else "junction"


# Stable colour per line category, shared across every region panel.
line_categories = sorted(line_category(edges).unique())
_line_palette = plt.get_cmap("tab10")
line_colour = {
    category: _line_palette(index % 10) for index, category in enumerate(line_categories)
}

# Every node is a bus; the kind only changes marker and colour.
NODE_STYLE = {
    "substation": {"color": "#111827", "marker": "o", "size": 26, "label": "substation bus"},
    "generator": {"color": "#7c3aed", "marker": "^", "size": 40, "label": "generator bus"},
    "junction": {"color": "#6b7280", "marker": ".", "size": 12, "label": "junction bus"},
}

regions = sorted(edges["region"].dropna().astype(str).unique())
if not regions:
    regions = [None]

figure, axes = plt.subplots(
    1,
    len(regions),
    figsize=(7 * len(regions), 7),
    squeeze=False,
)

for index, region in enumerate(regions):
    axis = axes[0, index]
    region_edges = edges if region is None else edges[edges["region"].eq(region)]
    region_nodes = nodes if region is None else nodes[nodes["region"].eq(region)]

    edge_category = line_category(region_edges)
    for category in line_categories:
        category_edges = region_edges[edge_category.eq(category)]
        if category_edges.empty:
            continue
        axis.add_collection(
            LineCollection(
                line_segments(category_edges),
                colors=[line_colour[category]],
                linewidths=1.1,
                alpha=0.85,
                label=category,
            )
        )

    if "kind" in region_nodes:
        kinds = region_nodes["kind"].map(node_kind)
        for kind, style in NODE_STYLE.items():
            kind_nodes = region_nodes[kinds.eq(kind)]
            if kind_nodes.empty:
                continue
            axis.scatter(
                kind_nodes.geometry.x,
                kind_nodes.geometry.y,
                s=style["size"],
                marker=style["marker"],
                color=style["color"],
                edgecolors="white",
                linewidths=0.4,
                zorder=3,
                label=style["label"],
            )

    set_map_extent(axis, region_edges)
    axis.set_title(region.title() if region is not None else OUTPUT_NAME)
    axis.legend(loc="best", fontsize=8)

figure.suptitle(
    f"{OUTPUT_NAME}: {len(network.buses):,} buses · {len(network.lines):,} lines "
    f"(lines coloured by {LINE_COLOUR_BY})"
)
figure.tight_layout()
VISUALISATION_DIR.mkdir(parents=True, exist_ok=True)
STATIC_MAP_PATH = VISUALISATION_DIR / f"{OUTPUT_NAME}-static-map.png"
figure.savefig(STATIC_MAP_PATH, dpi=180, bbox_inches="tight")
plt.close(figure)

print("Saved static map:", STATIC_MAP_PATH)
display(Image(filename=STATIC_MAP_PATH))


## Interactive map

The interactive map is shown inline below and also saved as a standalone HTML file with pan, zoom and an OpenStreetMap base layer. It uses the same GeoParquet geometry and the same source/voltage line colours as the static map.


In [ ]:
def flatten_lines(frame):
    longitudes = []
    latitudes = []
    for segment in line_segments(frame):
        longitudes.extend([point[0] for point in segment] + [None])
        latitudes.extend([point[1] for point in segment] + [None])
    return longitudes, latitudes


def hover_value(value):
    return "" if pd.isna(value) else str(value)


def rgba_to_hex(rgba):
    red, green, blue = (int(round(channel * 255)) for channel in rgba[:3])
    return f"#{red:02x}{green:02x}{blue:02x}"


interactive = go.Figure()

# One line trace per source/voltage category, using the static map's colours.
edge_category = line_category(edges)
for category in line_categories:
    category_edges = edges[edge_category.eq(category)]
    if category_edges.empty:
        continue
    longitudes, latitudes = flatten_lines(category_edges)
    interactive.add_trace(
        go.Scattermap(
            lon=longitudes,
            lat=latitudes,
            mode="lines",
            name=str(category),
            line={"color": rgba_to_hex(line_colour[category]), "width": 2},
            hoverinfo="skip",
        )
    )

# Mark the meaningful buses only. Junction buses are the road vertices and would
# add tens of thousands of markers, so they are left as line geometry here.
if "kind" in nodes:
    kinds = nodes["kind"].map(node_kind)
    for kind in ("substation", "generator"):
        style = NODE_STYLE[kind]
        kind_nodes = nodes[kinds.eq(kind)]
        if kind_nodes.empty:
            continue
        interactive.add_trace(
            go.Scattermap(
                lon=kind_nodes.geometry.x,
                lat=kind_nodes.geometry.y,
                mode="markers",
                name=style["label"],
                marker={"size": 9, "color": style["color"]},
                text=[
                    "<br>".join(
                        filter(
                            None,
                            (
                                str(row.bus_id),
                                hover_value(row.get("name")),
                                hover_value(row.get("kind")),
                                hover_value(row.get("source")),
                            ),
                        )
                    )
                    for _, row in kind_nodes.iterrows()
                ],
                hoverinfo="text",
            )
        )

min_x, min_y, max_x, max_y = edges.total_bounds
span = max(max_x - min_x, max_y - min_y, 0.01)
zoom = max(2.5, min(11.0, 8.0 - math.log2(span / 0.5)))
interactive.update_layout(
    title=f"{OUTPUT_NAME} (lines coloured by {LINE_COLOUR_BY})",
    height=720,
    margin={"l": 0, "r": 0, "t": 45, "b": 0},
    map={
        "style": "open-street-map",
        "center": {"lon": (min_x + max_x) / 2, "lat": (min_y + max_y) / 2},
        "zoom": zoom,
    },
    legend={"orientation": "h", "yanchor": "bottom", "y": 0.01},
)

VISUALISATION_DIR.mkdir(parents=True, exist_ok=True)
INTERACTIVE_MAP_PATH = VISUALISATION_DIR / f"{OUTPUT_NAME}-interactive-map.html"
interactive.write_html(
    INTERACTIVE_MAP_PATH,
    include_plotlyjs="cdn",
    full_html=True,
)

print("Saved interactive map:", INTERACTIVE_MAP_PATH)
display(FileLink(str(Path(os.path.relpath(INTERACTIVE_MAP_PATH, Path.cwd())))))

# Display the interactive map inline as well, not only as a saved file.
interactive.show()


## Saved outputs

In [ ]:
saved_outputs = pd.Series(
    {
        "result directory": RESULT_DIR,
        "canonical PyPSA network": NETWORK_PATH,
        "network metadata": METADATA_PATH,
        "GeoParquet nodes": NODES_PATH,
        "GeoParquet edges": EDGES_PATH,
        "spatial manifest": MANIFEST_PATH,
        "static map": STATIC_MAP_PATH,
        "interactive map": INTERACTIVE_MAP_PATH,
        "human review tables": HUMAN_TABLE_OUTPUT_ROOT / OUTPUT_NAME,
    },
    name="path",
)
display(saved_outputs.to_frame())

print(f"All modelling and visualisation artifacts for this result are under:\n{RESULT_DIR}")
for path in (
    NETWORK_PATH,
    METADATA_PATH,
    NODES_PATH,
    EDGES_PATH,
    MANIFEST_PATH,
    STATIC_MAP_PATH,
    INTERACTIVE_MAP_PATH,
):
    display(FileLink(str(Path(os.path.relpath(path, Path.cwd())))))